In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/us-used-cars-dataset/used_cars_data.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PowerTransformer, LabelEncoder, MinMaxScaler
from nltk.tokenize import word_tokenize
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
dataframe = pd.read_csv(r'/kaggle/input/us-used-cars-dataset/used_cars_data.csv')
dataframe

<ipython-input-3-cee4241b594a>:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(r'/kaggle/input/us-used-cars-dataset/used_cars_data.csv')
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/di

,vin,back_legroom,bed,bed_height,bed_length,body_type,cabin,city,city_fuel_economy,combine_fuel_economy,...,transmission,transmission_display,trimId,trim_name,vehicle_damage_category,wheel_system,wheel_system_display,wheelbase,width,year
0,ZACNJABB5KPJ92081,35.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,Bayamon,NaN,NaN,...,A,9-Speed Automatic Overdrive,t83804,Latitude FWD,NaN,FWD,Front-Wheel Drive,101.2 in,79.6 in,2019
1,SALCJ2FX1LH858117,38.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,9-Speed Automatic Overdrive,t86759,S AWD,NaN,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
2,JF1VA2M67G9829723,35.4 in,NaN,NaN,NaN,Sedan,NaN,Guaynabo,17.0,NaN,...,M,6-Speed Manual,t58994,Base,NaN,AWD,All-Wheel Drive,104.3 in,78.9 in,2016
3,SALRR2RV0L2433391,37.6 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,8-Speed Automatic Overdrive,t86074,V6 HSE AWD,NaN,AWD,All-Wheel Drive,115 in,87.4 in,2020
4,SALCJ2FXXLH862327,38.1 in,NaN,NaN,NaN,SUV / Crossover,NaN,San Juan,NaN,NaN,...,A,9-Speed Automatic Overdrive,t86759,S AWD,NaN,AWD,All-Wheel Drive,107.9 in,85.6 in,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000035,2GNAXJEV0J6261526,39.7 in,NaN,NaN,NaN,SUV / Crossover,NaN,Fairfield,26.0,NaN,...,A,Automatic,t72936,1.5T LT FWD,NaN,FWD,Front-Wheel Drive,107.3 in,72.6 in,2018
3000036,1GNERFKW0LJ225508,38.4 in,NaN,NaN,NaN,SUV / Crossover,NaN,Vallejo,18.0,NaN,...,A,Automatic,t85763,LS FWD,NaN,FWD,Front-Wheel Drive,120.9 in,78.6 in,2020
3000037,3FA6P0HD3GR134062,38.3 in,NaN,NaN,NaN,Sedan,NaN,Napa,NaN,NaN,...,A,6-Speed Automatic Overdrive,t57569,SE,NaN,FWD,Front-Wheel Drive,112.2 in,83.5 in,2016
3000038,SAJAJ4BNXHA968809,35 in,NaN,NaN,NaN,Sedan,NaN,Fairfield,30.0,NaN,...,A,Automatic,t65977,20d Premium AWD,NaN,AWD,All-Wheel Drive,111.6 in,81.7 in,2017


In [4]:
nulldict = dataframe.isnull().sum()
nulldict = nulldict > 2000000
print(nulldict)

vin                     False
back_legroom            False
bed                      True
bed_height               True
bed_length               True
                        ...  
wheel_system            False
wheel_system_display    False
wheelbase               False
width                   False
year                    False
Length: 66, dtype: bool


In [5]:
dic = dict(nulldict)
true_keys = [key for key in dic if dic[key]]
flattened_list = np.array(true_keys).flatten().tolist()
dataframe.drop(columns=flattened_list, axis=1, inplace=True)
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000040 entries, 0 to 3000039
Data columns (total 57 columns):
 #   Column                Dtype  
---  ------                -----  
 0   vin                   object 
 1   back_legroom          object 
 2   body_type             object 
 3   city                  object 
 4   city_fuel_economy     float64
 5   daysonmarket          int64  
 6   dealer_zip            object 
 7   description           object 
 8   engine_cylinders      object 
 9   engine_displacement   float64
 10  engine_type           object 
 11  exterior_color        object 
 12  fleet                 object 
 13  frame_damaged         object 
 14  franchise_dealer      bool   
 15  franchise_make        object 
 16  front_legroom         object 
 17  fuel_tank_volume      object 
 18  fuel_type             object 
 19  has_accidents         object 
 20  height                object 
 21  highway_fuel_economy  float64
 22  horsepower            float64
 23  interio

In [6]:
x = dataframe.nunique() > 1500000
x
dataframe.drop(columns=['vin', 'description', 'listing_id', 'main_picture_url'], axis=1, inplace=True)
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000040 entries, 0 to 3000039
Data columns (total 53 columns):
 #   Column                Dtype  
---  ------                -----  
 0   back_legroom          object 
 1   body_type             object 
 2   city                  object 
 3   city_fuel_economy     float64
 4   daysonmarket          int64  
 5   dealer_zip            object 
 6   engine_cylinders      object 
 7   engine_displacement   float64
 8   engine_type           object 
 9   exterior_color        object 
 10  fleet                 object 
 11  frame_damaged         object 
 12  franchise_dealer      bool   
 13  franchise_make        object 
 14  front_legroom         object 
 15  fuel_tank_volume      object 
 16  fuel_type             object 
 17  has_accidents         object 
 18  height                object 
 19  highway_fuel_economy  float64
 20  horsepower            float64
 21  interior_color        object 
 22  isCab                 object 
 23  is_new 

In [7]:
dataframe['back_legroom'] = dataframe['back_legroom'].astype(str)
dataframe['back_legroom'] = dataframe['back_legroom'].apply(lambda x: x.split()[0])
dataframe['wheelbase'] = dataframe['wheelbase'].astype(str)
dataframe['width'] = dataframe['width'].astype(str)
dataframe['wheelbase'] = dataframe['wheelbase'].apply(lambda x: x.split()[0])
dataframe['width'] = dataframe['width'].apply(lambda x: x.split()[0])

In [8]:
dataframe['torque'] = dataframe['torque'].astype(str)
dataframe['lb/ft'] = dataframe['torque'].apply(lambda x: x.split()[0])
dataframe['RPM'] = dataframe['torque'].apply(lambda x: x.split()[3] if isinstance(x, str) and len(x.split()) >= 4 else None)

In [9]:
dataframe.drop('torque', axis=1, inplace=True)

In [10]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000040 entries, 0 to 3000039
Data columns (total 54 columns):
 #   Column                Dtype  
---  ------                -----  
 0   back_legroom          object 
 1   body_type             object 
 2   city                  object 
 3   city_fuel_economy     float64
 4   daysonmarket          int64  
 5   dealer_zip            object 
 6   engine_cylinders      object 
 7   engine_displacement   float64
 8   engine_type           object 
 9   exterior_color        object 
 10  fleet                 object 
 11  frame_damaged         object 
 12  franchise_dealer      bool   
 13  franchise_make        object 
 14  front_legroom         object 
 15  fuel_tank_volume      object 
 16  fuel_type             object 
 17  has_accidents         object 
 18  height                object 
 19  highway_fuel_economy  float64
 20  horsepower            float64
 21  interior_color        object 
 22  isCab                 object 
 23  is_new 

In [11]:
dataframe['front_legroom'] = dataframe['front_legroom'].astype(str)
dataframe['fuel_tank_volume'] = dataframe['fuel_tank_volume'].astype(str)
dataframe['height'] = dataframe['height'].astype(str)
dataframe['maximum_seating'] = dataframe['maximum_seating'].astype(str)
dataframe['power'] = dataframe['power'].astype(str)

In [12]:
dataframe['front_legroom'] = dataframe['front_legroom'].apply(lambda x: x.split()[0])
dataframe['fuel_tank_volume'] = dataframe['fuel_tank_volume'].apply(lambda x: x.split()[0])
dataframe['height'] = dataframe['height'].apply(lambda x: x.split()[0])
dataframe['maximum_seating'] = dataframe['maximum_seating'].apply(lambda x: x.split()[0])

In [13]:
dataframe['power_hp'] = dataframe['power'].apply(lambda x: x.split()[0])
dataframe['power_rpm'] = dataframe['power'].apply(lambda x: x.split()[3] if isinstance(x, str) and len(x.split()) >= 4 else None)

In [14]:
dataframe.drop(columns=['trimId','sp_id','latitude','longitude','power'],axis=1,inplace=True)
dataframe

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,back_legroom,body_type,city,city_fuel_economy,daysonmarket,dealer_zip,engine_cylinders,engine_displacement,engine_type,exterior_color,...,trim_name,wheel_system,wheel_system_display,wheelbase,width,year,lb/ft,RPM,power_hp,power_rpm
0,35.1,SUV / Crossover,Bayamon,NaN,522,960,I4,1300.0,I4,Solar Yellow,...,Latitude FWD,FWD,Front-Wheel Drive,101.2,79.6,2019,200,"1,750",177,"5,750"
1,38.1,SUV / Crossover,San Juan,NaN,207,922,I4,2000.0,I4,Narvik Black,...,S AWD,AWD,All-Wheel Drive,107.9,85.6,2020,269,"1,400",246,"5,500"
2,35.4,Sedan,Guaynabo,17.0,1233,969,H4,2500.0,H4,NaN,...,Base,AWD,All-Wheel Drive,104.3,78.9,2016,290,"4,000",305,"6,000"
3,37.6,SUV / Crossover,San Juan,NaN,196,922,V6,3000.0,V6,Eiger Gray,...,V6 HSE AWD,AWD,All-Wheel Drive,115,87.4,2020,332,"3,500",340,"6,500"
4,38.1,SUV / Crossover,San Juan,NaN,137,922,I4,2000.0,I4,Narvik Black,...,S AWD,AWD,All-Wheel Drive,107.9,85.6,2020,269,"1,400",246,"5,500"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000035,39.7,SUV / Crossover,Fairfield,26.0,16,94533,I4,1500.0,I4,Silver,...,1.5T LT FWD,FWD,Front-Wheel Drive,107.3,72.6,2018,nan,None,nan,None
3000036,38.4,SUV / Crossover,Vallejo,18.0,171,94591,V6,3600.0,V6,Mosaic Black Metallic,...,LS FWD,FWD,Front-Wheel Drive,120.9,78.6,2020,266,"2,800",310,"6,800"
3000037,38.3,Sedan,Napa,NaN,91,94559,NaN,2000.0,NaN,Gray,...,SE,FWD,Front-Wheel Drive,112.2,83.5,2016,270,"1,750",240,"5,500"
3000038,35,Sedan,Fairfield,30.0,11,94533,I4 Diesel,2000.0,I4 Diesel,Green,...,20d Premium AWD,AWD,All-Wheel Drive,111.6,81.7,2017,318,"1,750",180,"4,000"


In [15]:
dataframe.drop(dataframe[dataframe['front_legroom'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['fuel_tank_volume'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['height'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['maximum_seating'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['power_hp'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['power_rpm'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['RPM'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['wheelbase'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['width'] == '--'].index, inplace=True)
dataframe.drop(dataframe[dataframe['back_legroom'] == '--'].index, inplace=True)

In [16]:
dataframe['power_rpm'] = dataframe['power_rpm'].str.replace(',', '')
dataframe['RPM'] = dataframe['RPM'].str.replace(',', '')

In [17]:
dataframe['front_legroom'] = dataframe['front_legroom'].astype('float64')
dataframe['fuel_tank_volume'] = dataframe['fuel_tank_volume'].astype('float64')
dataframe['height'] = dataframe['height'].astype('float64')
dataframe['maximum_seating'] = dataframe['maximum_seating'].astype('float64')
dataframe['power_hp'] = dataframe['power_hp'].astype('float64')
dataframe['power_rpm'] = dataframe['power_rpm'].astype('float64')
dataframe['RPM'] = dataframe['RPM'].astype('float64')
dataframe['wheelbase'] = dataframe['wheelbase'].astype('float64')
dataframe['width'] = dataframe['width'].astype('float64')
dataframe['back_legroom'] = dataframe['back_legroom'].astype('float64')

In [18]:
print(dataframe.dtypes)

back_legroom            float64
body_type                object
city                     object
city_fuel_economy       float64
daysonmarket              int64
dealer_zip               object
engine_cylinders         object
engine_displacement     float64
engine_type              object
exterior_color           object
fleet                    object
frame_damaged            object
franchise_dealer           bool
franchise_make           object
front_legroom           float64
fuel_tank_volume        float64
fuel_type                object
has_accidents            object
height                  float64
highway_fuel_economy    float64
horsepower              float64
interior_color           object
isCab                    object
is_new                     bool
length                   object
listed_date              object
listing_color            object
major_options            object
make_name                object
maximum_seating         float64
mileage                 float64
model_na

In [19]:
le = LabelEncoder()

catcol = dataframe.select_dtypes(include=['object'])

for x in catcol:
    dataframe[x] = dataframe[x].astype(str)

for x in catcol:
    dataframe[x] = le.fit_transform(dataframe[x])

In [20]:
pt = PowerTransformer(method='yeo-johnson')

for x in (dataframe.drop('price', axis=1)).columns:
    dataframe[x] = pt.fit_transform(dataframe[[x]])

/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:187: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


In [21]:
dataframe.skew()

back_legroom            0.066184
body_type               0.232206
city                   -0.274808
city_fuel_economy      -0.034758
daysonmarket           -0.004891
dealer_zip             -0.263472
engine_cylinders       -0.032498
engine_displacement     0.034023
engine_type            -0.032498
exterior_color         -0.244718
fleet                  -0.173242
frame_damaged           0.083275
franchise_dealer       -1.604156
franchise_make         -0.118887
front_legroom           0.246128
fuel_tank_volume       -0.066047
fuel_type               0.970030
has_accidents          -0.109337
height                  0.013275
highway_fuel_economy   -0.042381
horsepower              0.002954
interior_color         -0.094274
isCab                  -0.143664
is_new                  0.039595
length                  0.009994
listed_date            -0.688607
listing_color          -0.408121
major_options          -0.230558
make_name              -0.054916
maximum_seating        -0.247206
mileage   

In [ ]:
corr = dataframe.corr()

# 히트맵 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix Heatmap")
plt.show()

/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


In [ ]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2916078 entries, 0 to 3000039
Data columns (total 51 columns):
 #   Column                Dtype  
---  ------                -----  
 0   back_legroom          float64
 1   body_type             float64
 2   city                  float64
 3   city_fuel_economy     float64
 4   daysonmarket          float64
 5   dealer_zip            float64
 6   engine_cylinders      float64
 7   engine_displacement   float64
 8   engine_type           float64
 9   exterior_color        float64
 10  fleet                 float64
 11  frame_damaged         float64
 12  franchise_dealer      float64
 13  franchise_make        float64
 14  front_legroom         float64
 15  fuel_tank_volume      float64
 16  fuel_type             float64
 17  has_accidents         float64
 18  height                float64
 19  highway_fuel_economy  float64
 20  horsepower            float64
 21  interior_color        float64
 22  isCab                 float64
 23  is_new      

In [24]:
for x in dataframe.columns:
    dataframe[x].fillna(dataframe[x].mean(), inplace=True)

<ipython-input-24-0cd83845dbaf>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe[x].fillna(dataframe[x].mean(), inplace=True)


In [25]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2916078 entries, 0 to 3000039
Data columns (total 51 columns):
 #   Column                Dtype  
---  ------                -----  
 0   back_legroom          float64
 1   body_type             float64
 2   city                  float64
 3   city_fuel_economy     float64
 4   daysonmarket          float64
 5   dealer_zip            float64
 6   engine_cylinders      float64
 7   engine_displacement   float64
 8   engine_type           float64
 9   exterior_color        float64
 10  fleet                 float64
 11  frame_damaged         float64
 12  franchise_dealer      float64
 13  franchise_make        float64
 14  front_legroom         float64
 15  fuel_tank_volume      float64
 16  fuel_type             float64
 17  has_accidents         float64
 18  height                float64
 19  highway_fuel_economy  float64
 20  horsepower            float64
 21  interior_color        float64
 22  isCab                 float64
 23  is_new      

In [26]:
# 각 열의 결측치 개수 확인
missing_counts = dataframe.isnull().sum()

# 결측치 비율 확인
missing_percentage = (dataframe.isnull().sum() / len(dataframe)) * 100

# 결측치 데이터프레임 생성
missing_data = pd.DataFrame({
    'Missing Values': missing_counts,
    'Percentage': missing_percentage
}).sort_values(by='Missing Values', ascending=False)

print(missing_data)

                      Missing Values  Percentage
back_legroom                       0         0.0
theft_title                        0         0.0
make_name                          0         0.0
maximum_seating                    0         0.0
mileage                            0         0.0
model_name                         0         0.0
owner_count                        0         0.0
price                              0         0.0
salvage                            0         0.0
savings_amount                     0         0.0
seller_rating                      0         0.0
sp_name                            0         0.0
transmission                       0         0.0
listing_color                      0         0.0
transmission_display               0         0.0
trim_name                          0         0.0
wheel_system                       0         0.0
wheel_system_display               0         0.0
wheelbase                          0         0.0
width               

In [27]:
print(dataframe.describe(include='all'))

       back_legroom     body_type          city  city_fuel_economy  \
count  2.916078e+06  2.916078e+06  2.916078e+06       2.916078e+06   
mean  -5.514210e-16 -6.724342e-16 -6.612062e-17       5.476783e-15   
std    9.723080e-01  1.000000e+00  1.000000e+00       9.185051e-01   
min   -5.283116e+00 -2.836292e+00 -2.202763e+00      -5.830805e+00   
25%   -6.570932e-01 -1.833736e-02 -8.119631e-01      -6.675893e-01   
50%   -5.476481e-16 -1.833736e-02  7.664648e-02       5.477331e-15   
75%    5.311270e-01  9.513330e-01  8.868852e-01       6.227353e-01   
max    1.037566e+01  4.612090e+00  1.602390e+00       3.532085e+00   

       daysonmarket    dealer_zip  engine_cylinders  engine_displacement  \
count  2.916078e+06  2.916078e+06      2.916078e+06         2.916078e+06   
mean   6.923951e-17  2.395313e-16     -3.904859e-16        -1.588142e-15   
std    1.000000e+00  1.000000e+00      1.000000e+00         9.700607e-01   
min   -2.503040e+00 -2.262812e+00     -2.632679e+00        -4.043

In [28]:
print(dataframe.min())
print(dataframe.max())

back_legroom           -5.283116e+00
body_type              -2.836292e+00
city                   -2.202763e+00
city_fuel_economy      -5.830805e+00
daysonmarket           -2.503040e+00
dealer_zip             -2.262812e+00
engine_cylinders       -2.632679e+00
engine_displacement    -4.043517e+00
engine_type            -2.632679e+00
exterior_color         -2.372516e+00
fleet                  -1.148377e+00
frame_damaged          -9.606907e-01
franchise_dealer       -2.084002e+00
franchise_make         -2.575748e+00
front_legroom          -4.820932e+01
fuel_tank_volume       -1.073026e+01
fuel_type              -4.546146e+00
has_accidents          -1.091866e+00
height                 -3.101513e+00
highway_fuel_economy   -5.698248e+00
horsepower             -4.040940e+00
interior_color         -4.307323e+00
isCab                  -1.120996e+00
is_new                 -9.803984e-01
length                 -3.631320e+00
listed_date            -2.048598e+00
listing_color          -1.477623e+00
m

In [29]:
# 데이터 크기 확인
print(f"Rows: {dataframe.shape[0]}, Columns: {dataframe.shape[1]}")

# 샘플 데이터 확인
print(dataframe.head())

Rows: 2916078, Columns: 51
   back_legroom  body_type      city  city_fuel_economy  daysonmarket  \
0     -0.958835  -0.018337 -1.720647       5.477331e-15      2.207216   
1     -0.016913  -0.018337  1.005788       5.477331e-15      1.392370   
2     -0.869582   0.951333 -0.350284      -9.313063e-01      3.009301   
3     -0.181593  -0.018337  1.005788       5.477331e-15      1.345804   
4     -0.016913  -0.018337  1.005788       5.477331e-15      1.044630   

   dealer_zip  engine_cylinders  engine_displacement  engine_type  \
0    1.457589         -0.789735            -1.904396    -0.789735   
1    1.324458         -0.789735            -0.667188    -0.789735   
2    1.461680         -2.632679            -0.096180    -2.632679   
3    1.324458          0.863364             0.338281     0.863364   
4    1.324458         -0.789735            -0.667188    -0.789735   

   exterior_color  ...  trim_name  wheel_system  wheel_system_display  \
0        1.045333  ...  -0.032705      0.53306

In [32]:
dftrain = dataframe[:int(len(dataframe)/3)]

In [36]:
features = dftrain.drop('price', axis=1)
labels = dftrain['price']

features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.2, random_state=42)


In [37]:
xgb = XGBRegressor()

xgb.fit(features_train, labels_train)
pred = xgb.predict(features_test)

print('Accuracy:', xgb.score(features_test, labels_test))

Accuracy: 0.9297982161610598


In [38]:
dftest = dataframe[int(len(dataframe)/3)+1:]
print(len(dftest))

1944051


In [39]:
actualfeatures = dftest.drop('price', axis=1)
actual_answers = dftest['price']

predict = xgb.predict(actualfeatures)

In [40]:
mae = mean_absolute_error(predict, actual_answers)
mse = mean_squared_error(predict, actual_answers)
rmse = np.sqrt(mse)
r2 = r2_score(predict, actual_answers)

print("mean absolute error:", mae)
print("mean squared error:", mse)
print("root mean squared error:", rmse)
print("R-squared score :", r2)

mean absolute error: 2730.9792210114256
mean squared error: 48455193.111181885
root mean squared error: 6960.976448112857
R-squared score : 0.8263425973004946
